# M1_marker: klue/bert-base with typed entity markers

Fine-tunes klue/bert-base after wrapping each entity span with a typed marker (`@*TYPE*subject@`, `#^TYPE^object#`). The original project trained bert-base on plain text and scored 0.2042. Holding the backbone fixed and adding only markers measures the marker contribution on its own.

Config: lr 2e-5, batch 32, 5 epochs, seed 42. Built for a Kaggle T4 GPU; results land in `results/M1_marker/`.

In [1]:
import os, json, random, shutil, warnings
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import scipy.special

from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    EarlyStoppingCallback, DataCollatorWithPadding,
)
from sklearn.metrics import f1_score, average_precision_score, confusion_matrix
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")

SEED = 42
MAX_LENGTH = 256
NUM_LABELS = 30
NO_RELATION_ID = 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
RESULTS_DIR = os.path.join(BASE, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("cuda", torch.cuda.is_available(), "devices", torch.cuda.device_count())

cuda True devices 2


In [2]:
ds = load_dataset("klue/klue", "re")
train_ds, val_ds = ds["train"], ds["validation"]
label_names = val_ds.features["label"].names
val_labels = np.array(val_ds["label"])
print("train", len(train_ds), "val", len(val_ds), "labels", len(label_names))

README.md: 0.00B [00:00, ?B/s]

re/train-00000-of-00001.parquet:   0%|          | 0.00/6.65M [00:00<?, ?B/s]

re/validation-00000-of-00001.parquet:   0%|          | 0.00/1.54M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/32470 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7765 [00:00<?, ? examples/s]

train 32470 val 7765 labels 30


In [3]:
def add_markers(example):
    text = example["sentence"]
    subj, obj = example["subject_entity"], example["object_entity"]
    spans = [
        (subj["start_idx"], subj["end_idx"], subj["type"], "subj"),
        (obj["start_idx"], obj["end_idx"], obj["type"], "obj"),
    ]
    spans.sort(key=lambda s: s[0], reverse=True)
    for start, end, etype, role in spans:
        word = text[start:end + 1]
        tag = "@*" + etype + "*" + word + "@" if role == "subj" else "#^" + etype + "^" + word + "#"
        text = text[:start] + tag + text[end + 1:]
    return text


def micro_f1(labels, preds):
    mask = labels != NO_RELATION_ID
    if mask.sum() == 0:
        return 0.0
    return float(f1_score(labels[mask], preds[mask], average="micro", zero_division=0))


def auprc(labels, logits):
    probs = scipy.special.softmax(logits, axis=1)
    onehot = label_binarize(labels, classes=list(range(NUM_LABELS)))
    return float(average_precision_score(onehot, probs, average="micro"))


def metrics_fn(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"micro_f1": micro_f1(labels, preds), "auprc": auprc(labels, logits)}

In [4]:
def save_results(name, trainer, f1, ap, best_epoch, labels, preds):
    out = os.path.join(RESULTS_DIR, name)
    os.makedirs(out, exist_ok=True)
    with open(os.path.join(out, "metrics.json"), "w") as f:
        json.dump({"model": name, "micro_f1": f1, "auprc": ap, "best_epoch": best_epoch},
                  f, indent=2)

    cm = confusion_matrix(labels, preds, labels=list(range(NUM_LABELS)))
    short = [n[:10] for n in label_names]
    fig, ax = plt.subplots(figsize=(16, 14))
    im = ax.imshow(cm, cmap="Blues")
    plt.colorbar(im, ax=ax, fraction=0.03)
    ax.set_xticks(range(NUM_LABELS)); ax.set_xticklabels(short, rotation=90, fontsize=6)
    ax.set_yticks(range(NUM_LABELS)); ax.set_yticklabels(short, fontsize=6)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(name + " confusion matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(out, "confusion_matrix.png"), dpi=120, bbox_inches="tight")
    plt.close()

    logs = trainer.state.log_history
    tl = [(l["epoch"], l["loss"]) for l in logs if "loss" in l and "eval_loss" not in l]
    vl = [(l["epoch"], l["eval_loss"]) for l in logs if "eval_loss" in l]
    vf = [(l["epoch"], l["eval_micro_f1"]) for l in logs if "eval_micro_f1" in l]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    if tl: ax[0].plot(*zip(*tl), "b-o", ms=3, label="train_loss")
    if vl: ax[0].plot(*zip(*vl), "r-o", ms=3, label="val_loss")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].grid(alpha=0.3)
    if vf: ax[1].plot(*zip(*vf), "g-o", ms=3, label="val_micro_f1")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("micro_f1"); ax[1].legend(); ax[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(out, "loss_curve.png"), dpi=120, bbox_inches="tight")
    plt.close()
    print(name, "written to", out)

In [5]:
MODEL = "klue/bert-base"
tok = AutoTokenizer.from_pretrained(MODEL)

def encode(example):
    enc = tok(add_markers(example), max_length=MAX_LENGTH, truncation=True)
    enc["labels"] = example["label"]
    return enc

train_enc = train_ds.map(encode, remove_columns=train_ds.column_names)
val_enc = val_ds.map(encode, remove_columns=val_ds.column_names)

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Map:   0%|          | 0/32470 [00:00<?, ? examples/s]

Map:   0%|          | 0/7765 [00:00<?, ? examples/s]

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=NUM_LABELS, ignore_mismatched_sizes=True)
warmup = int((len(train_enc) // 32) * 5 * 0.1)

args = TrainingArguments(
    output_dir=os.path.join(BASE, "m1_marker_ckpt"),
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_steps=warmup,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    fp16=True,
    seed=SEED,
    data_seed=SEED,
    logging_steps=100,
    save_total_limit=2,
    report_to="none",
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_enc, eval_dataset=val_enc,
    processing_class=tok, data_collator=DataCollatorWithPadding(tok),
    compute_metrics=metrics_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

In [7]:
trainer.train()
pred = trainer.predict(val_enc)
preds = np.argmax(pred.predictions, axis=-1)
f1 = micro_f1(val_labels, preds)
ap = auprc(val_labels, pred.predictions)
best_epoch = max((l for l in trainer.state.log_history if "eval_micro_f1" in l),
                 key=lambda x: x["eval_micro_f1"])["epoch"]
print("M1_marker", "micro_f1", round(f1, 4), "auprc", round(ap, 4), "best_epoch", best_epoch)
print("marker contribution over plain bert-base 0.2042:", round(f1 - 0.2042, 4))
save_results("M1_marker", trainer, f1, ap, best_epoch, val_labels, preds)

Epoch,Training Loss,Validation Loss,Micro F1,Auprc
1,2.023709,2.343417,0.517230,0.668885
2,1.261844,1.767040,0.641353,0.763776
3,0.939663,1.627916,0.687939,0.790867
4,0.745699,1.555823,0.687939,0.822402
5,0.556007,1.627849,0.703893,0.815202


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[checkpoint load: klue/roberta LayerNorm gamma/beta renamed to weight/bias; classifier head newly initialised]


M1_marker micro_f1 0.7039 auprc 0.8152 best_epoch 5.0
marker contribution over plain bert-base 0.2042: 0.4997
M1_marker written to /kaggle/working/results/M1_marker
